# Italian-Spanish Pipeline Playground
Interactive notebook for testing individual pipeline functions.

In [ ]:
import sys, os
import pandas as pd

ROOT = os.path.abspath('.')
sys.path.insert(0, os.path.join(ROOT, 'pipeline', 'make-list'))
sys.path.insert(0, os.path.join(ROOT, 'pipeline', 'process-words'))
sys.path.insert(0, os.path.join(ROOT, 'pipeline', 'extract'))

## Load data

In [ ]:
# Stage 1 output (the main input file)
df = pd.read_csv(os.path.join(ROOT, 'input', '240510_Matthias-Buchmeier_Italian-frequency-list-1-5000.csv'))
df.head(10)

In [ ]:
# Stage 2 output (filtered list)
filtered = pd.read_csv(os.path.join(ROOT, 'temp', 'filtered_list.csv'))
filtered.head(10)

In [ ]:
# Final scored output
scored = pd.read_csv(os.path.join(ROOT, 'output', 'all_words_scored.csv'))
scored.head(10)

---
## Stage 1: Lemmatize

In [ ]:
from lemmatize import lemmatize

# Try lemmatizing a small sample
sample = pd.DataFrame({'rango': [1, 2, 3], 'parola': ['mangiando', 'correre', 'belle']})
lemmatize(sample)

## Stage 2: Deduplicate

In [ ]:
from deduplicate import deduplicate_lemmas

# See how deduplication works
sample = df.head(50).copy()
print(f'Before: {len(sample)}')
deduped = deduplicate_lemmas(sample)
print(f'After: {len(deduped)}')
deduped.head()

## Stage 2: POS Tagging

In [ ]:
from pos_tag import add_pos_tags

sample = df.head(20).copy()
tagged = add_pos_tags(sample)
tagged[['parola', 'lemma', 'pos']].head(10)

## Stage 2: Filters

In [ ]:
from filters import eliminate_proper_nouns, eliminate_symbols, eliminate_english_words

# Need POS tags first
sample = add_pos_tags(df.head(100).copy())

kept, elim_propn = eliminate_proper_nouns(sample)
print(f'Proper nouns removed: {len(elim_propn)}')

kept, elim_sym = eliminate_symbols(kept)
print(f'Symbols removed: {len(elim_sym)}')

kept, elim_en = eliminate_english_words(kept)
print(f'English words removed: {len(elim_en)}')

print(f'\nRemaining: {len(kept)} of {len(sample)}')

---
## Stage 3: Spanishify

In [ ]:
from spanishify import spanishify

examples = ['informazione', 'possibile', 'nazionale', 'parlare', 'partire', 'differenza', 'affetto']
for word in examples:
    print(f'{word:20s} → {spanishify(word)}')

## Stage 3: Similarity

In [ ]:
from similarity import remove_accents, normalized_levenshtein, get_synonyms, best_synonym_score

# Levenshtein examples
pairs = [('importante', 'importante'), ('parlare', 'hablar'), ('casa', 'casa'), ('cane', 'perro')]
for it, es in pairs:
    score = normalized_levenshtein(it, es)
    print(f'{it:15s} vs {es:15s} → {score:.3f}')

In [ ]:
# WordNet synonyms
words = ['casa', 'perro', 'hablar', 'tiempo']
for w in words:
    syns = get_synonyms(w, lang='spa')
    print(f'{w}: {syns}')

In [ ]:
# Best synonym score: Italian lemma vs Spanish synonyms
pairs = [('cane', 'perro'), ('tempo', 'tiempo'), ('acqua', 'agua')]
for it, es in pairs:
    score = best_synonym_score(it, es)
    print(f'{it:15s} vs {es:15s} → best synonym score: {score:.3f}')

## Stage 3: Score

In [ ]:
from score import score_all_words

# Score a subset of the filtered+translated data
sample = pd.read_csv(os.path.join(ROOT, 'temp', 'filtered_list.csv')).head(20)

# Need Spanish translations — load from cache if available
spanish_cache = os.path.join(ROOT, 'temp', 'spanish.csv')
if os.path.exists(spanish_cache):
    spanish = pd.read_csv(spanish_cache)
    sample = sample.merge(spanish[['parola', 'spagnolo']], on='parola', how='left')

    from clean_spanish import clean_and_lemmatize
    sample = clean_and_lemmatize(sample)
    scored_sample = score_all_words(sample)
    scored_sample[['parola', 'lemma', 'spagnolo', 'lemma_es', 'match_type', 'score']]
else:
    print('No Spanish translation cache found. Run Stage 3 first.')

---
## Explore the final results

In [ ]:
scored = pd.read_csv(os.path.join(ROOT, 'output', 'all_words_scored.csv'))

print(f'Total words: {len(scored)}')
print(f'\nScore distribution:')
print(scored['score'].describe())
print(f'\nMatch types:')
print(scored['match_type'].value_counts())

In [ ]:
# Top scoring pairs
scored.nlargest(20, 'score')[['parola', 'spagnolo', 'match_type', 'score']]

In [ ]:
# Lowest scoring pairs (hardest words for Spanish speakers)
scored.nsmallest(20, 'score')[['parola', 'spagnolo', 'match_type', 'score']]